# UMUD R80T — krawędzie rozcięgien z zespołu map (Colab, GPU, bez treningu)

**Po co.** Sondy LB pokazały, że FL zależy od położenia krawędzi rozcięgien: cecha FL z mapy A76 uśrednionej z v2
dała **B8 = 0,20650** (−0,00084 względem B7). Ten notebook sprawdza, czy jeszcze lepsze krawędzie da się uzyskać
**bez trenowania** — z modeli, które już są na Drive:

1. **R12B z odbiciem poziomym (TTA)** — R12B był jedynym modelem rozcięgien liczonym bez TTA.
2. **Multi-scale TTA** dla v2 (672 / 768 / 864 px) i A76 (896 / 1024 / 1152 px), każde także z odbiciem.
3. **Średnie map** trzech modeli (R12B, v2, A76) w kilku wariantach.

Każda mapa jest ograniczona do **tej samej pary rozcięgien co R12B** (strażnik pary z R76T) — zmienia się tylko
położenie krawędzi, nigdy wybór pary. Dla każdego obrazu testu i OSF powstają linie krawędzi (jak `apo_pair_record`)
i pomiary MT (jak `apo_pair_subpixel`) w 10 konfiguracjach:

| konfiguracja | co to jest |
|---|---|
| `r12b`, `v2`, `a76_safe`, `a76_v2mix` | **kontrole** — muszą odtworzyć eksport R76T |
| `r12b_tta` | R12B + odbicie |
| `a76ms_safe` | A76, 3 skale × odbicie |
| `mix2ms` | średnia A76 i v2, obie multi-scale (wersja `a76_v2mix` z multi-scale) |
| `mix3` | średnia R12B + A76 + v2 |
| `mix3t` | jw., R12B z odbiciem |
| `mix3ms` | średnia R12B z odbiciem + A76 multi-scale + v2 multi-scale |

**Uruchomienie:** Colab → *Środowisko wykonawcze → Zmień typ* → GPU (L4 / A100, ewentualnie T4) → **Uruchom wszystko**,
zatwierdź dostęp do Drive. Czas na GPU nie był mierzony; szacuję 20–40 minut (344 obrazy, ~70 przebiegów sieci na obraz).
Postęp zapisuje się po każdym obrazie — po rozłączeniu uruchom ten sam plik ponownie.

**Wynik:** `UMUD_Challenge/outputs/R80T_v1_0_0/R80T_wyniki.zip` (kilkaset KB). **Prześlij ten plik w czacie.**
Cechy FL policzę z niego dokładnie tak jak w B8 (te same fragmenty R22 i F71) i przygotuję sondy.

Potrzebne na Drive (jak w R76T): wagi `UMUD_R12B/best_dinov2_frozen.pt`, 5 foldów v2 (`outputs/R45T_v1_0_0/apo2`),
5 foldów A76 (`outputs/R76T_v1_0_0/A76`), obrazy testu i OSF z manifestów oraz eksport R76T (do kontroli).


In [ ]:
# 1. USTAWIENIA — zwykle nic nie zmieniasz.
RUN_NAME = 'R80T_v1_0_0'          # ten sam katalog = wznowienie
A76_RUN = 'R76T_v1_0_0'           # skąd brać foldy A76 i eksport kontrolny
MS_SCALES = (0.875, 1.0, 1.125)   # skale multi-scale TTA (v2: 672/768/864 px, A76: 896/1024/1152 px)
CHECK_IMAGE_SHA = True            # sprawdza, czy obrazy są dokładnie te z manifestów
SESSION_HOURS = 3.0               # po tym czasie zapis i bezpieczne zatrzymanie
RUN_SELF_TEST = True


In [ ]:
# 2. ŚRODOWISKO I SAMOWYSTARCZALNY KOD (GPL-3.0-or-later). Nic tu nie trenuje.
import os, sys, time, json, hashlib, base64, zlib, subprocess, importlib.util, zipfile
from pathlib import Path
T_START = time.time()
FIXTURE = os.environ.get('R80T_FIXTURE_ROOT')   # tylko lokalne testy autora
if FIXTURE:
    DRIVE_ROOT = Path(FIXTURE) / 'MyDrive'; LOCAL = Path(FIXTURE) / 'content'; LOCAL.mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive'); LOCAL = Path('/content')
PROJECT = DRIVE_ROOT / 'UMUD_Challenge'; RUN = PROJECT / 'outputs' / RUN_NAME; RUN.mkdir(parents=True, exist_ok=True)
PACKAGE_SHA = 'b0fa541e7abed1300c2d8f77d4bd4e437bb5125705187f52d171b0a56f33e916'
PAYLOAD = '...'
raw = zlib.decompress(base64.b64decode(PAYLOAD))
if hashlib.sha256(raw).hexdigest() != PACKAGE_SHA:
    raise RuntimeError('Uszkodzony kod notebooka — wczytaj oryginalny plik ponownie')
SRC = LOCAL / ('r80t_src_' + PACKAGE_SHA[:12]); SRC.mkdir(parents=True, exist_ok=True)
for rel, content in json.loads(raw).items():
    path = SRC / rel
    if not path.resolve().is_relative_to(SRC.resolve()):
        raise ValueError('Invalid source path')
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(content)
for module, package in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('cv2', 'opencv-python-headless'),
                        ('skimage', 'scikit-image'), ('timm', 'timm')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('torchvision') is None:
    raise RuntimeError('Potrzebne torch + torchvision. Wybierz standardowe środowisko GPU Colab.')
sys.path.insert(0, str(SRC))
for name in ['r45_core', 'r45_models', 'r45_decode', 'r45_apo', 'r45_infer', 'r71_geom', 'r71_train', 'r76_data',
             'r76_train', 'r76_export', 'r76_selftest', 'r80_ens']:
    sys.modules.pop(name, None)
import numpy as np, pandas as pd, torch, torchvision, cv2
import r45_core, r45_infer, r80_ens
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and not FIXTURE:
    raise RuntimeError('Brak GPU. Colab: Środowisko wykonawcze → Zmień typ → GPU.')
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (testy)'
def log(*args):
    msg = ' '.join(str(a) for a in args); print(msg, flush=True)
    with (RUN / 'LOG.txt').open('a') as f:
        f.write(time.strftime('%Y-%m-%d %H:%M:%S ') + msg + '\n')
def show(frame, n=40):
    print(frame.head(n).to_string(index=False) if len(frame) else '(pusto)', flush=True)
code_dir = RUN / 'code'; code_dir.mkdir(exist_ok=True)
archive = code_dir / f'R80T_source_{PACKAGE_SHA[:12]}.zip'
if not archive.exists():
    tmp = archive.with_suffix('.tmp.zip')
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for rel, content in json.loads(raw).items():
            z.writestr(rel, content)
    tmp.replace(archive)
r45_core.atomic_json(RUN / 'ENVIRONMENT.json', {'python': sys.version, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
    'numpy': np.__version__, 'pandas': pd.__version__, 'opencv': cv2.__version__, 'gpu': GPU, 'cuda': torch.version.cuda,
    'package_sha256': PACKAGE_SHA})
ASSETS = SRC / 'assets'
DEADLINE = T_START + SESSION_HOURS * 3600
if FIXTURE and os.environ.get('R80T_V2_SIDE'):
    r80_ens.V2_SIDE = int(os.environ['R80T_V2_SIDE'])
log('R80T |', GPU, '| kod', PACKAGE_SHA[:12], '|', RUN)


In [ ]:
# 3. WEJŚCIA: istniejące wagi (każda sprawdzana SHA-256), manifesty obrazów, eksport R76T do kontroli.
if RUN_SELF_TEST:
    r45_core.atomic_json(RUN / 'SELF_TEST.json', r80_ens.selftest(log))
R12B_SHA = os.environ.get('R80T_R12B_SHA', r80_ens.R12B_SHA) if FIXTURE else r80_ens.R12B_SHA
REG = r80_ens.registry(PROJECT, DRIVE_ROOT, A76_RUN, R12B_SHA)
r45_core.atomic_json(RUN / 'REGISTRY.json', REG)
test_csv = Path(os.environ.get('R80T_TEST_MANIFEST', ASSETS / 'test_manifest.csv')) if FIXTURE else ASSETS / 'test_manifest.csv'
osf_csv = Path(os.environ.get('R80T_OSF_MANIFEST', ASSETS / 'osf_manifest.csv')) if FIXTURE else ASSETS / 'osf_manifest.csv'
MANIFESTS = {'test': r45_infer.load_manifest(test_csv, PROJECT, CHECK_IMAGE_SHA),
             'osf': r45_infer.load_manifest(osf_csv, PROJECT, CHECK_IMAGE_SHA)}
R76_EXPORT = None
latest = PROJECT / 'outputs' / A76_RUN / 'LATEST_EXPORT.json'
if latest.is_file():
    d = Path(json.loads(latest.read_text())['directory'])
    if d.is_dir():
        R76_EXPORT = d
if R76_EXPORT is None:
    found = sorted((PROJECT / 'outputs' / A76_RUN / 'exports').glob('A5_*/SUMMARY.json'), key=lambda p: p.stat().st_mtime)
    R76_EXPORT = found[-1].parent if found else None
log('Wagi: R12B, v2 5/5, A76 5/5 (bok', REG['a76_side'], 'px) | obrazy: test', len(MANIFESTS['test']), 'OSF', len(MANIFESTS['osf']))
log('Eksport kontrolny R76T:', R76_EXPORT if R76_EXPORT else 'nie znaleziono — kontrola i pomocnicze FL zostaną pominięte')


In [ ]:
# 4. EKSPORT — wznawialny po każdym obrazie.
BUDGET_HIT = False
try:
    OUT, FRAMES = r80_ens.export(RUN, REG, MANIFESTS, PACKAGE_SHA, DEVICE, MS_SCALES, DEADLINE, log)
except r80_ens.SessionBudgetReached as e:
    BUDGET_HIT = True; log(str(e), '— uruchom wszystko ponownie w nowej sesji (ten sam RUN_NAME)')


In [ ]:
# 5. RAPORTY I PACZKA DO CZATU.
if not BUDGET_HIT:
    FL = r80_ens.fl_tables(OUT, R76_EXPORT, MANIFESTS, FRAMES, log) if R76_EXPORT else {}
    CTRL = r80_ens.control_report(OUT, R76_EXPORT, FRAMES) if R76_EXPORT else pd.DataFrame()
    OSF = r80_ens.osf_report(OUT, ASSETS / 'osf35_full.csv', FRAMES['osf'], FL.get('osf'))
    CHG = r80_ens.change_report(OUT, FRAMES, FL)
    r45_core.atomic_json(OUT / 'SUMMARY.json', {'status': 'R80T_EXPORT_COMPLETE', 'version': r80_ens.VERSION, 'gpu': GPU,
        'rows': {k: len(v) for k, v in FRAMES.items()}, 'configs': list(r80_ens.CONFIGS), 'ms_scales': list(MS_SCALES),
        'r76t_export': str(R76_EXPORT) if R76_EXPORT else None, 'minutes': round((time.time() - T_START) / 60, 1),
        'public_score': 'NOT_MEASURED', 'note': 'Bez treningu. Para rozcięgien jak R12B; zmieniają się tylko krawędzie.'})
    ZIP = r80_ens.pack(OUT, RUN, extra=[RUN / 'LOG.txt', RUN / 'ENVIRONMENT.json', RUN / 'REGISTRY.json', RUN / 'SELF_TEST.json'])
    print('\nKONTROLA względem R76T (max różnica krawędzi w px; MT w mm):'); show(CTRL)
    print('\nZMIANY względem kontroli r12b na teście (MT mm, FL %):'); show(CHG)
    print('\nOSF (pomocniczo; średnia 7 ekspertów):'); show(OSF)
    log('GOTOWE. Prześlij w czacie plik:', ZIP, f'({ZIP.stat().st_size / 1024:.0f} KB)')
else:
    log('Bieg zapisany. W nowej sesji: ten sam notebook → Uruchom wszystko.')


## Co dalej

- **`CONTROL_R76T.csv`** — kontrole `r12b`, `v2`, `a76_safe`, `a76_v2mix` powinny pokrywać się z eksportem R76T
  (różnice krawędzi rzędu 0,01 px wynikają tylko z arytmetyki GPU). Duże różnice oznaczają inne wagi albo obrazy — wtedy nie używamy wyników.
- **`CHANGES_test.csv`** — jak bardzo nowe konfiguracje przesuwają MT i FL względem `r12b` (FL liczone tu pomocniczo
  z fragmentów eksportu R76T).
- **`OSF_R80.csv`** — porównanie z 7 ekspertami OSF; to tylko wskazówka (OSF kilka razy mylił się co do LB).

Po przesłaniu `R80T_wyniki.zip` policzę cechy FL dokładnie jak w B8 (te same fragmenty R22 i F71, ta sama reguła lex5)
i przygotuję notebook Kaggle z sondami. Wybór między konfiguracjami rozstrzygnie LB, nie OSF.
